# Simple Linear Regression Capstone Exercise: Restaurant Tips

Author: *Your name here*

**About this notebook.** This is a scaffolded exercise that asks you to apply *every* idea from the six SLR notebooks (`17_1_1` through `17_1_6`) to a brand-new dataset: the classic `tips` dataset from `seaborn`, which records 244 restaurant bills and the tip the server received on each.

**Your scientific question:**

> *Does the size of the bill predict the size of the tip — and how confidently can we say so?*

**What you will do.** Six Parts, each mirroring one of the SLR notebooks:

| Part | Concept | Mirrors |
| :--- | :--- | :--- |
| 1 | Fitting an SLR with scikit-learn and statsmodels | `17_1_1` |
| 2 | Significance: standard errors, p-values, CIs (via simulation) | `17_1_2` |
| 3 | The LINE assumptions: diagnosing the model | `17_1_3` |
| 4 | Influence, leverage, Cook's Distance | `17_1_4` |
| 5 | Transformations | `17_1_5` |
| 6 | Train / test split and the bias–variance tradeoff | `17_1_6` |

**How to use this notebook.** Most of the code is given to you. Cells marked **`# YOUR TURN`** contain fill-in-the-blank tasks. Markdown cells marked **Reflection** ask you to write a 1–3 sentence answer. If you get stuck on a step, open the corresponding SLR notebook and look up how the same operation was done there.

**Data:** `seaborn.load_dataset('tips')` — 244 rows describing meals at one restaurant. We will use `total_bill` (in dollars) to predict `tip` (in dollars).


## 0. Setup

Run the cell below to import everything we will use across the six Parts.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
rng = np.random.default_rng(seed=42)


### 0.1 Load and look at the data

> **Rule from the preliminaries series:** always look at the data before modeling it.


In [ ]:
tips = sns.load_dataset('tips').reset_index(drop=True)
print(f"Rows: {len(tips)}")
tips.head()


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(tips['total_bill'], tips['tip'], s=25, alpha=0.55, color='steelblue')
ax.set_xlabel('Total bill ($)')
ax.set_ylabel('Tip ($)')
ax.set_title('Restaurant tips: total bill vs. tip')
plt.show()


**A first look.** A positive trend is obvious — larger bills get larger tips. But look at the *spread*: at low bills the tips hug a narrow band; at high bills the tips fan out from about \$2 to \$10. Hold onto that observation; it will come back in Part 3 when we check the *Equal Variance* assumption.


---
# Part 1 — The Two Paradigms (sklearn vs. statsmodels)

Mirrors `17_1_1_SLR_SKLearn_vs_Statsmodel.ipynb`.

You will fit the *same* simple linear regression with both libraries and verify that they give identical slope, intercept, and $R^2$ — the only difference is what each tool *reports* back to you.


### 1.1 Scikit-learn fit &nbsp; *(YOUR TURN)*

Recall from `17_1_1` that scikit-learn's `LinearRegression` requires **`X` to be a 2D matrix**, even when there's only one feature. The classic fix is `.reshape(-1, 1)`.

* Reshape `total_bill` into a 2D `X`.
* Keep `tip` as a 1D `y`.
* Fit a `LinearRegression`.
* Pull out `.coef_[0]`, `.intercept_`, and `.score(X, y)` (which returns $R^2$).


In [ ]:
# YOUR TURN: build X (2D) and y (1D), then fit
X = tips['total_bill'].values.reshape(___)
y = tips['tip'].values

model_sk = ___
model_sk.fit(X, y)

slope_sk     = model_sk.coef_[0]
intercept_sk = model_sk.intercept_
r2_sk        = model_sk.score(X, y)

print(f"Scikit-learn slope:     {slope_sk:.4f}  $ of tip per $ of bill")
print(f"Scikit-learn intercept: {intercept_sk:.4f}  $")
print(f"Scikit-learn R^2:       {r2_sk:.4f}")


### 1.2 Statsmodels fit &nbsp; *(YOUR TURN)*

Now fit the same regression with **statsmodels**. Two reminders from `17_1_1`:

1. Statsmodels does **not** add an intercept for you — you must do `X_sm = sm.add_constant(x)`.
2. The argument order is reversed: `sm.OLS(y, X)`, not `(X, y)`.


In [ ]:
x_series = tips['total_bill']
y_series = tips['tip']

# YOUR TURN: add a constant column and fit OLS
X_sm = ___
model_sm = ___.fit()

print(model_sm.summary())


### 1.3 Cross-check: do both libraries agree?


In [ ]:
slope_sm     = model_sm.params['total_bill']
intercept_sm = model_sm.params['const']
r2_sm        = model_sm.rsquared

print(f"Statsmodels slope:      {slope_sm:.4f}")
print(f"Statsmodels intercept:  {intercept_sm:.4f}")
print(f"Statsmodels R^2:        {r2_sm:.4f}")
print()
print(f"Slopes match?     {np.isclose(slope_sk, slope_sm)}")
print(f"Intercepts match? {np.isclose(intercept_sk, intercept_sm)}")
print(f"R^2 values match? {np.isclose(r2_sk, r2_sm)}")


**Reflection 1.** In one sentence, what does the slope mean *in real-world terms* (not just statistically)? Is the intercept meaningful here — i.e., does `total_bill = 0` correspond to a sensible scenario?

*Your answer:*


---
# Part 2 — Significance: Did We Just Get Lucky?

Mirrors `17_1_2_SLR_Significance.ipynb`.

The summary table from Part 1 reports a standard error, t-statistic, p-value, and 95% CI for the slope. In this Part you will **rebuild each of those numbers from scratch via simulation** and confirm they agree with what statsmodels reports.


### 2.1 Read off the inferential numbers from the summary table

These are the numbers we will try to reproduce by simulation.


In [ ]:
observed_slope = model_sm.params['total_bill']
se_summary     = model_sm.bse['total_bill']
t_summary      = model_sm.tvalues['total_bill']
p_summary      = model_sm.pvalues['total_bill']
ci_low_summary, ci_high_summary = model_sm.conf_int().loc['total_bill']

print(f"slope:          {observed_slope:.4f}")
print(f"std err:        {se_summary:.4f}")
print(f"t-statistic:    {t_summary:.3f}")
print(f"p-value:        {p_summary:.3g}")
print(f"95% CI:         [{ci_low_summary:.3f}, {ci_high_summary:.3f}]")


### 2.2 Bootstrap the standard error &nbsp; *(YOUR TURN)*

The standard error tells us how much the slope would *wiggle* across different samples of 244 meals. We can simulate that wiggle by **resampling rows with replacement** and recording the slope each time.

* Use `rng.integers(0, N, size=N)` to draw row indices with replacement.
* Recompute the slope using the closed-form $\hat{\beta}_1 = \mathrm{Cov}(x, y) / \mathrm{Var}(x)$ each iteration.
* Save 2,000 bootstrap slopes.


In [ ]:
x_arr = tips['total_bill'].to_numpy()
y_arr = tips['tip'].to_numpy()
N = len(tips)
n_iter = 2000

bootstrap_slopes = np.empty(n_iter)
for i in range(n_iter):
    # YOUR TURN: draw row indices with replacement, then compute the slope
    idx = ___
    xb, yb = x_arr[idx], y_arr[idx]
    bootstrap_slopes[i] = np.cov(xb, yb, bias=True)[0, 1] / np.var(xb)

print(f"Bootstrap SE (std of {n_iter} slopes): {bootstrap_slopes.std(ddof=1):.4f}")
print(f"Statsmodels SE from the summary:        {se_summary:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(bootstrap_slopes, bins=40, color='steelblue', alpha=0.75, edgecolor='white')
ax.axvline(bootstrap_slopes.mean(), color='darkorange', linewidth=2,
           label=f'mean = {bootstrap_slopes.mean():.3f}')
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='null: slope = 0')
ax.set_xlabel('Slope across bootstrap samples')
ax.set_ylabel('count')
ax.set_title(f'Bootstrap distribution of the slope ({n_iter} resamples)')
ax.legend()
plt.show()


**A small surprise.** In `17_1_2`, the bootstrap SE matched the statsmodels SE almost exactly (1.49 vs. 1.54 g/mm for Palmer Penguins). Here you should see the bootstrap SE come out *noticeably larger* than the statsmodels SE — maybe 50%+ wider. This is **not** a bug. The statsmodels formula assumes the LINE assumptions all hold; the bootstrap makes no such assumption. When the two disagree, that disagreement is itself a clue that *some* LINE assumption is shaky. Make a mental note — Part 3 will find the culprit.


### 2.3 Permutation p-value

The bootstrap tells us how much the slope wiggles *if our relationship is real*. The **permutation test** does the opposite: it destroys any relationship between `total_bill` and `tip` by shuffling `y`, and asks how often a "null world" produces a slope as extreme as ours.


In [ ]:
n_perm = 5000
null_slopes = np.empty(n_perm)
for i in range(n_perm):
    y_shuffled = rng.permutation(y_arr)
    null_slopes[i] = np.cov(x_arr, y_shuffled, bias=True)[0, 1] / np.var(x_arr)

p_empirical = np.mean(np.abs(null_slopes) >= np.abs(observed_slope))

print(f"Empirical p-value (permutation):  {p_empirical:.4g}  (out of {n_perm} shuffles)")
print(f"Statsmodels p-value (from table): {p_summary:.4g}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(null_slopes, bins=40, color='steelblue', alpha=0.75, edgecolor='white',
        label='null-world slopes (y shuffled)')
ax.axvline(observed_slope, color='red', linewidth=2,
           label=f'our slope = {observed_slope:.3f}')
ax.axvline(-observed_slope, color='red', linewidth=2, linestyle=':')
ax.set_xlabel('Slope under the null hypothesis')
ax.set_ylabel('count')
ax.set_title('Null distribution: our observed slope is nowhere on this plot')
ax.legend()
plt.show()


### 2.4 95% confidence interval — two ways &nbsp; *(YOUR TURN)*

* **Bootstrap percentile method:** take the 2.5th and 97.5th percentiles of `bootstrap_slopes`.
* **Formula:** $\hat{\beta}_1 \pm t^* \cdot \text{SE}$, where $t^*$ is the critical value from a t-distribution with `model_sm.df_resid` degrees of freedom.


In [ ]:
# YOUR TURN: compute the bootstrap percentile CI
ci_boot_low, ci_boot_high = np.percentile(bootstrap_slopes, ___)

# YOUR TURN: compute the formula-based CI
t_crit = stats.t.ppf(___, df=model_sm.df_resid)   # two-sided 95% critical value
ci_formula_low  = observed_slope - t_crit * se_summary
ci_formula_high = observed_slope + t_crit * se_summary

print(f"Bootstrap percentile CI: [{ci_boot_low:.4f}, {ci_boot_high:.4f}]")
print(f"Formula CI (t* * SE):    [{ci_formula_low:.4f}, {ci_formula_high:.4f}]")
print(f"Statsmodels CI:          [{ci_low_summary:.4f}, {ci_high_summary:.4f}]")


**Reflection 2.** A microscopic p-value is sometimes accompanied by an effect size so small it barely matters in practice. Is the relationship here *practically* significant? Translate the slope into a real-world sentence (e.g., *"on a \$100 bill, a server can expect roughly \$___ in tip"*) — and decide whether that's a meaningful effect or noise dressed up as a t-statistic.

*Your answer:*


---
# Part 3 — The LINE Assumptions

Mirrors `17_1_3_SLR_LINE_Assumptions.ipynb`.

> **OLS will draw a line through anything. It doesn't care whether a line is a good idea.**

Every inferential number from Part 2 (SE, p-value, CI) is built on the four **LINE** assumptions. We now check them in order.

| Letter | Assumption | How we check it |
|:-:|---|---|
| **L** | Linearity | Residuals vs. fitted, with a lowess smoother |
| **I** | Independence | Durbin–Watson (from the summary) — flags time-series autocorrelation |
| **N** | Normality of residuals | Q–Q plot |
| **E** | Equal Variance | Residual plot — look for a funnel |


### 3.1 L — Linearity

A residual plot should look like a horizontal band of noise around zero. Any curve in the lowess line means OLS has missed structure.


In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

resid  = model_sm.resid
fitted = model_sm.fittedvalues
smooth = lowess(resid, fitted, frac=0.4)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(fitted, resid, s=22, alpha=0.5, color='steelblue')
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.plot(smooth[:, 0], smooth[:, 1], color='red', linewidth=2.5, label='lowess')
ax.set_xlabel(r'Fitted $\hat{y}$ (predicted tip)')
ax.set_ylabel('Residual ($)')
ax.set_title('Residuals vs. Fitted — Linearity check')
ax.legend()
plt.show()


### 3.2 I — Independence

For cross-sectional data like this (one row per meal, no time ordering), independence is usually safe. The **Durbin–Watson** statistic at the bottom of the summary tests for autocorrelation among consecutive residuals. Values near 2 mean independence; values much below 1.7 or much above 2.3 are warnings.


In [ ]:
print(f"Durbin–Watson: {sm.stats.stattools.durbin_watson(model_sm.resid):.3f}")


### 3.3 N — Normality of residuals


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sm.qqplot(model_sm.resid, line='s', ax=ax,
          markerfacecolor='steelblue', markeredgecolor='steelblue', alpha=0.5)
for line in ax.get_lines():
    if line.get_linestyle() != 'None':
        line.set_color('red'); line.set_linewidth(2)
ax.set_title('Q–Q plot of residuals')
plt.show()


### 3.4 E — Equal variance &nbsp; *(YOUR TURN)*

Look again at the residuals-vs-fitted plot from 3.1, but this time pay attention to the **vertical spread** as you move left-to-right. A funnel (megaphone shape) is a violation of equal variance.

As a quantitative check, compute the standard deviation of residuals for the *bottom-half* of fitted values vs. the *top-half*. If the ratio is well above 1, the spread is growing.


In [ ]:
midpoint = np.median(fitted)

# YOUR TURN: residuals in the lower half of fitted vs. upper half
low_half_resid  = resid[fitted < ___]
high_half_resid = resid[fitted >= ___]

print(f"Std of residuals in the bottom 50%: {low_half_resid.std():.3f}")
print(f"Std of residuals in the top 50%:    {high_half_resid.std():.3f}")
print(f"Ratio (top / bottom):                {high_half_resid.std() / low_half_resid.std():.2f}")


**Reflection 3.** For each LINE assumption (L, I, N, E), write one sentence: *"This assumption looks {satisfied / violated / marginal} because ___."* Then state which assumption is the biggest concern for this dataset.

*Your answer:*


---
# Part 4 — Influence, Leverage, and Cook's Distance

Mirrors `17_1_4_SLR_Influence.ipynb`.

In `17_1_4` a synthetic "poisoned" row was injected into Ames Housing to give Cook's Distance an unambiguous villain to flag. We will do the same here: a single fake meal that has wildly inflated leverage and a wildly mis-positioned tip.

Three terms (from `17_1_4`):

| Term | Meaning |
|---|---|
| **Leverage** | Unusual $x$-value |
| **Outlier** | Unusual $y$-value given $x$ (large residual) |
| **Influence** | Removing the point would meaningfully move the line. |

A point becomes truly damaging when it has **high leverage *and* a large residual**.


### 4.1 Inject one poisoned row

A fictional \$200 bill that left a \$1 tip. This is far outside the natural range of the data — perfect Cook's-Distance bait.


In [ ]:
tips_poisoned = pd.concat([
    tips[['total_bill', 'tip']],
    pd.DataFrame({'total_bill': [200.0], 'tip': [1.0]}),
], ignore_index=True)

print(f"Rows after poisoning: {len(tips_poisoned)}")

X_p = sm.add_constant(tips_poisoned['total_bill'])
y_p = tips_poisoned['tip']
model_p = sm.OLS(y_p, X_p).fit()

print(f"\nBefore poisoning: slope = {observed_slope:.4f},  R² = {model_sm.rsquared:.4f}")
print(f"After poisoning:  slope = {model_p.params['total_bill']:.4f},  R² = {model_p.rsquared:.4f}")


### 4.2 Compute leverage, studentized residuals, and Cook's Distance


In [ ]:
influence = model_p.get_influence()

leverages   = influence.hat_matrix_diag
stud_resids = influence.resid_studentized_external
cooks_d, _  = influence.cooks_distance

print(f"Maximum leverage:                 {leverages.max():.4f}")
print(f"Maximum |studentized residual|:   {np.abs(stud_resids).max():.2f}")
print(f"Maximum Cook's Distance:           {cooks_d.max():.3f}")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].stem(leverages, markerfmt=' ', basefmt=' ', linefmt='steelblue')
axes[0].set_ylabel('Leverage  $h_{ii}$')
axes[0].set_title('Leverage (weirdness in $x$)')

axes[1].stem(stud_resids, markerfmt=' ', basefmt=' ', linefmt='steelblue')
axes[1].axhline(3, color='red', linestyle='--', linewidth=1, label='|value| = 3')
axes[1].axhline(-3, color='red', linestyle='--', linewidth=1)
axes[1].set_ylabel('Studentized residual')
axes[1].set_title('Studentized residuals (weirdness in $y$)')
axes[1].legend()

threshold = 4 / len(tips_poisoned)
axes[2].stem(cooks_d, markerfmt=' ', basefmt=' ', linefmt='steelblue')
axes[2].axhline(threshold, color='red', linestyle='--', linewidth=1,
                label=f'4/n = {threshold:.4f}')
axes[2].set_yscale('log')
axes[2].set_ylabel("Cook's Distance (log)")
axes[2].set_xlabel('Row index')
axes[2].set_title("Cook's Distance — one spike dominates")
axes[2].legend()

plt.tight_layout()
plt.show()


### 4.3 Identify the suspect &nbsp; *(YOUR TURN)*

Find the index of the row with the largest Cook's Distance and print its `(total_bill, tip)` values.


In [ ]:
# YOUR TURN: find the index of the maximum Cook's Distance
suspect_idx = ___

print(f"Suspect row index: {suspect_idx}")
print(f"Suspect (total_bill, tip): "
      f"({tips_poisoned.loc[suspect_idx, 'total_bill']}, {tips_poisoned.loc[suspect_idx, 'tip']})")


### 4.4 The drop test &nbsp; *(YOUR TURN)*

Refit the model **without** the suspect row and compare the slope, intercept, and $R^2$ to the poisoned fit. How much did one row change everything?


In [ ]:
# YOUR TURN: drop the suspect row, refit
tips_clean = tips_poisoned.drop(index=suspect_idx).reset_index(drop=True)

X_c = sm.add_constant(tips_clean['total_bill'])
y_c = tips_clean['tip']
model_c = ___.fit()

print(f"{'':22s}{'slope':>10s}  {'R²':>8s}")
print(f"{'Poisoned model':22s}{model_p.params['total_bill']:>10.4f}  {model_p.rsquared:>8.4f}")
print(f"{'After dropping suspect':22s}{model_c.params['total_bill']:>10.4f}  {model_c.rsquared:>8.4f}")
print(f"{'Original (Part 1)':22s}{observed_slope:>10.4f}  {model_sm.rsquared:>8.4f}")


### 4.5 Huber regression — a robust alternative

Sometimes you cannot ethically drop a row (no documented data-entry error, just an unusual case). **Huber regression** down-weights observations with large residuals instead of deleting them. Let's apply it to the *poisoned* data and see what slope it recovers.


In [ ]:
X_huber = tips_poisoned[['total_bill']].to_numpy()
y_huber = tips_poisoned['tip'].to_numpy()

huber = HuberRegressor().fit(X_huber, y_huber)

print(f"OLS on poisoned data:     slope = {model_p.params['total_bill']:.4f}")
print(f"Huber on poisoned data:   slope = {huber.coef_[0]:.4f}")
print(f"OLS after dropping row:   slope = {model_c.params['total_bill']:.4f}")
print(f"OLS on original (Part 1): slope = {observed_slope:.4f}")


**Reflection 4.** Cook's Distance fingered one row. Did dropping it *fully* restore the original slope, or only partway? And — given how cleanly Huber recovers the original — write one sentence on when you'd reach for robust regression instead of deletion.

*Your answer:*


---
# Part 5 — Transformations: Bending the Data, Not the Line

Mirrors `17_1_5_SLR_Transformations.ipynb`.

Part 3 flagged the funnel in the residuals (the **E**qual-variance assumption is shaky). The classic fix when the residuals fan out as $\hat{y}$ grows: take the log of `y`. We will:

1. Use **Box–Cox** to ask the data which transformation it wants.
2. Fit a **log–level** model: $\log(\text{tip}) = \beta_0 + \beta_1 \cdot \text{total\_bill}$.
3. Recheck the residual plot to see if the funnel is gone.
4. Interpret the new slope.

> Working with the un-poisoned `tips` (Part 1 data) for the rest of this Part.


### 5.1 Box–Cox on `tip` &nbsp; *(YOUR TURN)*

Recall from `17_1_5`:

| $\lambda$ | suggests |
| :-: | --- |
| 1.0 | no transformation needed |
| 0.5 | square root |
| 0.0 | log |
| −1.0 | reciprocal |


In [ ]:
# YOUR TURN: run scipy.stats.boxcox on the tip column
_, lam_tip = stats.boxcox(___)

print(f"Box–Cox lambda for `tip`: {lam_tip:.3f}")
print()
if abs(lam_tip) < 0.25:
    print("  → near 0: suggests a LOG transformation of y")
elif abs(lam_tip - 0.5) < 0.2:
    print("  → near 0.5: suggests a SQRT transformation of y")
elif abs(lam_tip - 1) < 0.2:
    print("  → near 1: y is already well-behaved")


### 5.2 Fit a log–level model


In [ ]:
tips_t = tips.copy()
tips_t['log_tip'] = np.log(tips_t['tip'])

X_log = sm.add_constant(tips_t['total_bill'])
model_log = sm.OLS(tips_t['log_tip'], X_log).fit()

print(model_log.summary().tables[1])
print(f"\nR² (log–level model): {model_log.rsquared:.3f}")


### 5.3 Recheck the residual plot


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(model_sm.fittedvalues, model_sm.resid, s=22, alpha=0.5, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel(r'Fitted $\hat{y}$ (raw model)')
axes[0].set_ylabel('Residual')
axes[0].set_title('Raw model: funnel present')

axes[1].scatter(model_log.fittedvalues, model_log.resid, s=22, alpha=0.5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel(r'Fitted $\hat{y}$ (log–level model)')
axes[1].set_ylabel('Residual (log scale)')
axes[1].set_title('log(tip) ~ total_bill: funnel reduced')

plt.tight_layout()
plt.show()


### 5.4 Interpreting the log–level slope

Recall the table from `17_1_5`:

| Model | Slope says |
| :--- | :--- |
| level–level | +1 unit of $x$ ⇒ +$\beta_1$ units of $y$ |
| **log–level** | +1 unit of $x$ ⇒ +$100 \cdot \beta_1$ % change in $y$ |
| level–log | +1% in $x$ ⇒ +$\beta_1/100$ units of $y$ |
| log–log | +1% in $x$ ⇒ +$\beta_1$% change in $y$ (pure elasticity) |


In [ ]:
beta_log = model_log.params['total_bill']
print(f"log–level slope: {beta_log:.5f}")
print()
print(f"A $1 increase in the bill is associated with roughly a "
      f"{beta_log * 100:.2f}% increase in the tip.")
print(f"A $10 increase in the bill →   {(np.exp(beta_log * 10) - 1) * 100:.1f}% increase in tip.")
print(f"A $20 increase in the bill →   {(np.exp(beta_log * 20) - 1) * 100:.1f}% increase in tip.")


**Reflection 5.** Compare the two residual plots in 5.3. Did the log transform fix the funnel? *Caution from `17_1_5`*: do **not** compare the raw and log models by $R^2$ — they predict different scales. The right comparison is residual *patterns*.

*Your answer:*


---
# Part 6 — The Generalization Test (Train / Test Split)

Mirrors `17_1_6_SLR_GeneralizationTest.ipynb`.

Every $R^2$ so far has measured how well our line predicts the *same* meals it was fit on — which is the most optimistic possible score. The honest question is: *how well would this line predict tips at a meal it has never seen?*

The answer is the **train/test split**.


### 6.1 A single 80/20 split &nbsp; *(YOUR TURN)*


In [ ]:
X = tips[['total_bill']].to_numpy()
y = tips['tip'].to_numpy()

# YOUR TURN: split 80/20 with random_state=42
X_train, X_test, y_train, y_test = train_test_split(___)

m = LinearRegression().fit(X_train, y_train)
train_r2 = m.score(X_train, y_train)
test_r2  = m.score(X_test,  y_test)

print(f"Train R²: {train_r2:.4f}")
print(f"Test  R²: {test_r2:.4f}")
print(f"Gap:      {train_r2 - test_r2:+.4f}")


### 6.2 Many splits — is the gap systematic? &nbsp; *(YOUR TURN)*

A *single* split is noisy. Repeat the split 200 times and plot the distribution of the train–test gap.


In [ ]:
n_splits = 200
gaps = np.empty(n_splits)
test_scores = np.empty(n_splits)

for i in range(n_splits):
    # YOUR TURN: split with random_state=i and fit
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=___)
    m = LinearRegression().fit(X_tr, y_tr)
    test_scores[i] = m.score(X_te, y_te)
    gaps[i] = m.score(X_tr, y_tr) - m.score(X_te, y_te)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(test_scores, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(test_scores.mean(), color='darkorange', linestyle='--', linewidth=2,
                label=f'mean = {test_scores.mean():.3f}')
axes[0].set_xlabel('Test $R^2$')
axes[0].set_ylabel('count')
axes[0].set_title(f'Test $R^2$ across {n_splits} random splits')
axes[0].legend()

axes[1].hist(gaps, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('train $R^2$ − test $R^2$')
axes[1].set_ylabel('count')
axes[1].set_title('Gap is centered near zero')

plt.tight_layout()
plt.show()

print(f"Mean gap: {gaps.mean():+.4f}   Std gap: {gaps.std():.4f}")


### 6.3 Making overfitting visible: polynomial features

A straight line with two parameters cannot memorize much, so it cannot overfit. To *see* overfitting we add powers of `total_bill` — `total_bill`, `total_bill²`, …, `total_bill^d` — and watch the train and test $R^2$ diverge as `d` grows.


In [ ]:
degrees = list(range(1, 13))
train_r2_list = []
test_r2_list = []

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

for d in degrees:
    model = make_pipeline(PolynomialFeatures(degree=d), LinearRegression())
    model.fit(X_tr, y_tr)
    train_r2_list.append(model.score(X_tr, y_tr))
    test_r2_list.append(model.score(X_te, y_te))

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(degrees, train_r2_list, marker='o', label='Train $R^2$', color='steelblue')
ax.plot(degrees, test_r2_list,  marker='o', label='Test $R^2$',  color='darkorange')
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('$R^2$')
ax.set_ylim(-1, 1.05)
ax.set_title('Bias–variance tradeoff: simple → overfit')
ax.legend()
plt.show()

print("Tabulated:")
for d, tr, te in zip(degrees, train_r2_list, test_r2_list):
    print(f"  degree {d:>2d}:  train R² = {tr:>+.3f}   test R² = {te:>+.3f}")


**Reflection 6.** Look at the two curves above.

1. At what degree does the test $R^2$ peak — and what does that tell us about the right complexity for *this* dataset?
2. Past that peak, the train $R^2$ keeps climbing while the test $R^2$ falls (sometimes going negative). Explain in one sentence what the model is doing differently to the training data at degree 10+ than at degree 2.

*Your answer:*


---
# Part 7 — Choosing Between Two Predictors

Parts 1–6 used `total_bill` as the sole predictor of `tip`. Before moving to multiple regression, a natural question is: *is there a better single predictor?*

The `tips` dataset also has `size` — the number of people at the table. Intuitively, larger parties might tip more (more food, more drink). In this Part you will:

1. Fit SLR with `size` as the predictor.
2. Compare its slope, significance, and $R^2$ to the `total_bill` model from Part 1.
3. Make a recommendation about which predictor to carry into MLR.

### 7.1 Fit SLR with `size` as predictor &nbsp; *(YOUR TURN)*

Repeat the fitting procedure from Part 1 using `size` instead of `total_bill`. Fit both sklearn and statsmodels.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns
from sklearn.linear_model import LinearRegression

tips = sns.load_dataset('tips').reset_index(drop=True)

# YOUR TURN: reshape size into a 2D X and fit sklearn
X_size = tips['size'].values.reshape(___)
y      = tips['tip'].values

sk_size = ___
sk_size.fit(X_size, y)

slope_size     = sk_size.coef_[0]
intercept_size = sk_size.intercept_
r2_size        = sk_size.score(X_size, y)

print(f'size → tip:  slope = {slope_size:.4f}  intercept = {intercept_size:.4f}  R² = {r2_size:.4f}')

### 7.2 Statsmodels summary for `size`

In [ ]:
# YOUR TURN: build X_sm_size (add a constant to the size column) and fit OLS
X_sm_size = ___
sm_size   = ___.fit()
print(sm_size.summary())

### 7.3 Side-by-side comparison

In [ ]:
# Given — re-fit the total_bill model for the comparison table
X_bill = tips[['total_bill']].to_numpy()
sk_bill = LinearRegression().fit(X_bill, y)
X_sm_bill = sm.add_constant(tips['total_bill'])
sm_bill   = sm.OLS(y, X_sm_bill).fit()

comparison = pd.DataFrame({
    'Predictor':   ['total_bill', 'size'],
    'Slope':       [sk_bill.coef_[0],          slope_size],
    'p-value':     [sm_bill.pvalues['total_bill'], sm_size.pvalues['size']],
    'R²':          [sk_bill.score(X_bill, y),   r2_size],
})
comparison = comparison.set_index('Predictor')
print(comparison.to_string())

**Reflection 7.** Answer the three questions below in 1–2 sentences each.

1. Which predictor has a higher $R^2$? What does that tell you about which one is a better standalone predictor?
2. Both slopes are positive. Translate each slope into a plain-English statement (e.g., *"each additional dollar on the bill is associated with $X more tip"*).
3. Given that `total_bill` and `size` both matter, would you expect a multiple regression model using **both** predictors to do better than either alone? What risk do you need to watch for when combining predictors?

*Your answers:*

---
# Wrap-up

You just walked the entire simple linear regression pipeline end-to-end on a fresh dataset:

1. **Two paradigms** — fit the same regression with scikit-learn and statsmodels, and verified they agree on the math even though they differ in what they report.
2. **Significance** — rebuilt the standard error (bootstrap), p-value (permutation), and 95% CI (percentile + formula) from scratch and matched statsmodels.
3. **LINE assumptions** — diagnosed Linearity, Independence, Normality, and Equal Variance, and identified which one was the weakest for this dataset.
4. **Influence** — used Cook's Distance to pinpoint a single damaging row, dropped it, and compared the result to a robust (Huber) refit.
5. **Transformations** — used Box–Cox to motivate a `log(y)` transformation and reinterpreted the slope in the new units.
6. **Generalization** — confirmed that a simple line generalizes well via repeated train/test splits, and made overfitting visible with polynomial features.

If any step feels unfamiliar, the matching SLR notebook (`17_1_1` through `17_1_6`) walks the same operation through in detail.
